# Moving Average (MA) Models

A **moving average model** of order $q$, written MA($q$), expresses the
current value as a linear combination of past **forecast errors** (white
noise terms):

$$
y_t = c + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \theta_2 \varepsilon_{t-2} + \cdots + \theta_q \varepsilon_{t-q}
$$

where $\varepsilon_t \sim \text{WN}(0, \sigma^2)$.

**Critical distinction**: this is NOT the same as moving average smoothing.
Moving average smoothing computes a rolling mean of *observed values*.  The
MA model uses past *forecast errors* (innovations).

This notebook covers:
1. MA(1) and MA(2) — theory, simulation, and ACF patterns
2. Backshift notation for MA
3. Stationarity and invertibility
4. Using the ACF to identify the order $q$
5. Fitting MA models with `ARIMA(order=(0,0,q))`

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")
rng = np.random.default_rng(42)

---
## 1. MA vs Moving Average Smoothing

Before diving in, let us be absolutely clear about the distinction:

| | Moving Average Smoothing | MA($q$) Model |
|---|---|---|
| Inputs | Past **observed values** $y_{t-1}, y_{t-2}, \ldots$ | Past **forecast errors** $\varepsilon_{t-1}, \varepsilon_{t-2}, \ldots$ |
| Purpose | Smooth out noise to reveal trend | Model the autocorrelation structure |
| Weights | Equal weights (simple) or specified | Estimated coefficients $\theta_1, \ldots, \theta_q$ |
| Usage | Data preparation / decomposition | Forecasting model component |

---
## 2. The MA(1) Model

$$
y_t = c + \varepsilon_t + \theta_1 \varepsilon_{t-1}
$$

Each observation is the current shock plus a fraction $\theta_1$ of the
previous shock.  The process has a **memory of exactly one period** — $y_t$
is correlated with $y_{t-1}$ but not with $y_{t-2}, y_{t-3}, \ldots$

Properties:
- $\text{Var}(y_t) = \sigma^2(1 + \theta_1^2)$
- $\rho_1 = \theta_1 / (1 + \theta_1^2)$
- $\rho_k = 0$ for $k \geq 2$

In [ ]:
def simulate_ma(theta_list, c=0, n=500, seed=42):
    """Simulate an MA(q) process.

    Parameters
    ----------
    theta_list : list of float
        MA coefficients [theta_1, theta_2, ...].
    c : float
        Constant (mean of the process).
    n : int
        Length of the series.
    seed : int
        Random seed.
    """
    q = len(theta_list)
    rng_local = np.random.default_rng(seed)
    eps = rng_local.standard_normal(n + q)  # extra q for burn-in
    y = np.zeros(n)
    for t in range(n):
        y[t] = c + eps[t + q]
        for j, theta in enumerate(theta_list, start=1):
            y[t] += theta * eps[t + q - j]
    return y


# Simulate MA(1) with different theta values
theta_values = [0.8, -0.8, 0.3, -0.3]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, theta in zip(axes.flat, theta_values):
    y = simulate_ma([theta], n=300)
    ax.plot(y, linewidth=0.8)
    ax.axhline(0, color="grey", linestyle="--", alpha=0.5)
    ax.set_title(f"MA(1) with $\\theta_1 = {theta}$")
    ax.set_xlabel("t")

plt.suptitle("Simulated MA(1) Processes", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ACF and PACF of MA(1)
y_ma1 = simulate_ma([0.7], n=1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y_ma1, lags=20, ax=axes[0], title="ACF of MA(1), $\\theta_1=0.7$")
plot_pacf(y_ma1, lags=20, ax=axes[1], title="PACF of MA(1), $\\theta_1=0.7$", method="ywm")
plt.tight_layout()
plt.show()

print("ACF: single significant spike at lag 1, then cuts off to zero.")
print("PACF: exponential/geometric decay.")
print("\nThis is the OPPOSITE pattern from AR!")
print("  AR(p):  PACF cuts off after lag p, ACF decays")
print("  MA(q):  ACF cuts off after lag q, PACF decays")

---
## 3. The MA(2) Model

$$
y_t = c + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \theta_2 \varepsilon_{t-2}
$$

The ACF will have significant spikes at lags 1 and 2, then cut off.

In [ ]:
y_ma2 = simulate_ma([0.6, 0.4], n=1000)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(y_ma2[:200], linewidth=0.8)
axes[0].set_title("MA(2): $\\theta_1=0.6, \\theta_2=0.4$")
axes[0].set_xlabel("t")
axes[0].axhline(0, color="grey", linestyle="--", alpha=0.5)

plot_acf(y_ma2, lags=20, ax=axes[1], title="ACF")
plot_pacf(y_ma2, lags=20, ax=axes[2], title="PACF", method="ywm")

plt.tight_layout()
plt.show()

print("ACF has significant spikes at lags 1 and 2, then cuts off.")
print("This tells us the true order is q=2.")

---
## 4. Backshift Notation for MA

Using the backshift operator $B$, an MA($q$) model is:

$$
y_t = c + (1 + \theta_1 B + \theta_2 B^2 + \cdots + \theta_q B^q) \varepsilon_t
$$

or compactly: $y_t = c + \Theta(B) \varepsilon_t$, where
$\Theta(B) = 1 + \theta_1 B + \theta_2 B^2 + \cdots + \theta_q B^q$ is the
**MA characteristic polynomial**.

---
## 5. Stationarity and Invertibility

### Stationarity

An MA($q$) process is **always stationary**, regardless of the values of
$\theta_1, \ldots, \theta_q$.  Why?  Because it is a finite linear
combination of white noise terms — both its mean and variance are finite
and constant over time.

### Invertibility

An MA process is **invertible** if all roots of $\Theta(B) = 0$ lie outside
the unit circle.  Invertibility means the MA can be rewritten as an infinite
AR process:

$$
\varepsilon_t = y_t - \theta_1 y_{t-1} - \theta_1^2 y_{t-2} - \cdots
$$

For MA(1): invertibility requires $|\theta_1| < 1$.

Invertibility ensures a **unique** MA representation and is needed for
parameter estimation.

In [ ]:
# Demonstrate: MA(1) with theta=0.7 and theta=1/0.7 produce the SAME ACF
# Only the invertible one (|theta|<1) is preferred

theta_a = 0.7
theta_b = 1 / 0.7  # = 1.4286 (non-invertible)

# Theoretical rho_1 for MA(1)
rho1_a = theta_a / (1 + theta_a**2)
rho1_b = theta_b / (1 + theta_b**2)

print(f"MA(1) with theta = {theta_a}:")
print(f"  rho_1 = {rho1_a:.6f}")
print(f"  Invertible: {abs(theta_a) < 1}")
print()
print(f"MA(1) with theta = {theta_b:.4f}:")
print(f"  rho_1 = {rho1_b:.6f}")
print(f"  Invertible: {abs(theta_b) < 1}")
print()
print("Both have the same lag-1 autocorrelation!")
print("We choose the invertible representation (|theta| < 1) by convention.")

In [ ]:
# Visual confirmation: both produce nearly identical ACFs
y_inv = simulate_ma([theta_a], n=2000, seed=99)
y_noninv = simulate_ma([theta_b], n=2000, seed=99)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y_inv, lags=15, ax=axes[0], title=f"ACF — MA(1), $\\theta={theta_a}$ (invertible)")
plot_acf(y_noninv, lags=15, ax=axes[1], title=f"ACF — MA(1), $\\theta={theta_b:.2f}$ (non-invertible)")
plt.tight_layout()
plt.show()

print("Both ACFs look very similar — the invertibility constraint")
print("resolves this ambiguity by selecting the unique representation.")

---
## 6. ACF Identifies the MA Order

For a pure MA($q$) process:
- ACF is **nonzero** at lags $1, 2, \ldots, q$
- ACF is **zero** for all lags $> q$ (it "cuts off" after lag $q$)

This is the mirror image of the PACF rule for AR:

| Model | ACF | PACF |
|-------|-----|------|
| AR($p$) | Decays | Cuts off after lag $p$ |
| MA($q$) | Cuts off after lag $q$ | Decays |

In [ ]:
# Demonstrate ACF cutoff for MA(1), MA(2), MA(3)
y_ma1_demo = simulate_ma([0.6], n=1000, seed=10)
y_ma2_demo = simulate_ma([0.6, 0.4], n=1000, seed=10)
y_ma3_demo = simulate_ma([0.5, 0.3, 0.2], n=1000, seed=10)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, y, title in zip(axes, [y_ma1_demo, y_ma2_demo, y_ma3_demo],
                         ["MA(1)", "MA(2)", "MA(3)"]):
    plot_acf(y, lags=15, ax=ax, title=f"ACF — {title}")

plt.tight_layout()
plt.show()

print("MA(1) ACF: cuts off after lag 1.")
print("MA(2) ACF: cuts off after lag 2.")
print("MA(3) ACF: cuts off after lag 3.")

---
## 7. Fitting MA Models with statsmodels

There is no standalone MA class in statsmodels.  Instead, use the ARIMA
class with `order=(0, 0, q)` — this specifies zero AR terms, zero
differencing, and $q$ MA terms:

```python
from statsmodels.tsa.arima.model import ARIMA
model = ARIMA(endog, order=(0, 0, q))
result = model.fit()
```

In [ ]:
# Fit MA(2) to simulated data and verify parameter recovery
y_sim = simulate_ma([0.6, 0.4], n=500, seed=77)
y_series = pd.Series(y_sim, name="y")

model = ARIMA(y_series, order=(0, 0, 2))
result = model.fit()
print(result.summary())

In [ ]:
# Compare estimated vs true parameters
print("True parameters:")
print(f"  theta_1 = 0.6")
print(f"  theta_2 = 0.4")
print(f"  c       = 0.0")
print()
print("Estimated parameters:")
print(f"  theta_1 = {result.params['ma.L1']:.4f}")
print(f"  theta_2 = {result.params['ma.L2']:.4f}")
print(f"  c       = {result.params['const']:.4f}")
print()
print("The estimates are close to the true values.")

In [ ]:
# In-sample fitted values
fitted = result.fittedvalues

fig, ax = plt.subplots()
ax.plot(y_series[:100], label="Actual", alpha=0.7)
ax.plot(fitted[:100], label="Fitted (MA(2))", alpha=0.7, linestyle="--")
ax.set_title("MA(2) — Actual vs Fitted (first 100 obs)")
ax.legend()
ax.set_xlabel("t")
plt.tight_layout()
plt.show()

In [ ]:
# Out-of-sample forecast
forecast_obj = result.get_forecast(steps=24)
fc_mean = forecast_obj.predicted_mean
fc_ci = forecast_obj.conf_int(alpha=0.05)

fig, ax = plt.subplots()
ax.plot(y_series.iloc[-80:], label="Actual")
ax.plot(fc_mean.index, fc_mean.values, label="Forecast", color="tab:red", linestyle="--")
ax.fill_between(fc_ci.index, fc_ci.iloc[:, 0], fc_ci.iloc[:, 1],
                color="tab:red", alpha=0.15, label="95% CI")
ax.axvline(y_series.index[-1], color="grey", linestyle=":", alpha=0.7)
ax.set_title("MA(2) — Forecast with Prediction Intervals")
ax.legend()
plt.tight_layout()
plt.show()

print("MA(q) forecasts beyond horizon q collapse to the mean.")
print("This is because MA(q) has a memory of only q periods.")
print(f"After step q={2}, the forecast equals the estimated constant: {result.params['const']:.4f}")

---
## 8. Comparing AR and MA Signatures

The diagnostic table is fundamental to ARIMA modelling:

| | ACF | PACF |
|---|---|---|
| **AR($p$)** | Exponential / sinusoidal decay | Cuts off after lag $p$ |
| **MA($q$)** | Cuts off after lag $q$ | Exponential / sinusoidal decay |
| **ARMA($p$,$q$)** | Decays (no sharp cutoff) | Decays (no sharp cutoff) |

When both ACF and PACF decay without a clear cutoff, the data may need
a mixed ARMA model — which leads us to ARIMA in the next notebook.

In [ ]:
# Side-by-side: AR(2) vs MA(2) — observe the mirror pattern
def simulate_ar2(phi1, phi2, n=1000, seed=42):
    rng_local = np.random.default_rng(seed)
    eps = rng_local.standard_normal(n)
    y = np.zeros(n)
    for t in range(2, n):
        y[t] = phi1 * y[t-1] + phi2 * y[t-2] + eps[t]
    return y

ar2_data = simulate_ar2(0.5, 0.3, n=1000, seed=55)
ma2_data = simulate_ma([0.5, 0.3], n=1000, seed=55)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(ar2_data, lags=15, ax=axes[0, 0], title="AR(2) — ACF (decays)")
plot_pacf(ar2_data, lags=15, ax=axes[0, 1], title="AR(2) — PACF (cuts off at 2)", method="ywm")
plot_acf(ma2_data, lags=15, ax=axes[1, 0], title="MA(2) — ACF (cuts off at 2)")
plot_pacf(ma2_data, lags=15, ax=axes[1, 1], title="MA(2) — PACF (decays)", method="ywm")

plt.suptitle("AR vs MA: Mirror-Image ACF/PACF Patterns", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Residual Check on the Fitted MA Model

In [ ]:
residuals = result.resid

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(residuals, linewidth=0.5)
axes[0].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[0].set_title("Residuals")

plot_acf(residuals, lags=20, ax=axes[1], title="ACF of Residuals")

axes[2].hist(residuals, bins=25, edgecolor="black", alpha=0.7, density=True)
axes[2].set_title("Residual Histogram")

plt.tight_layout()
plt.show()

# Ljung-Box test
from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(residuals, lags=[10, 15, 20], return_df=True)
print("Ljung-Box test:")
print(lb)
print("\np-values > 0.05 indicate white noise residuals — model is adequate.")

---
## Summary

- **MA($q$)** models express $y_t$ as a linear combination of past forecast
  errors: $y_t = c + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \cdots + \theta_q \varepsilon_{t-q}$.
- This is **NOT** the same as moving average smoothing — MA models use past
  *errors*, not past *observations*.
- MA processes are **always stationary** (finite sum of white noise).
- **Invertibility** ($|\theta| < 1$ for MA(1); roots of $\Theta(B)$ outside
  the unit circle in general) ensures a unique representation.
- The **ACF cuts off after lag $q$** for a pure MA($q$) — this is the
  diagnostic for MA order selection.
- In backshift notation: $y_t = c + \Theta(B) \varepsilon_t$.
- In statsmodels, fit MA models with `ARIMA(endog, order=(0, 0, q))`.
- MA forecasts beyond $q$ steps ahead equal the estimated constant — the
  model has finite memory.

In [ ]:
print("Next notebook: ARIMA — combining AR, differencing, and MA into a")
print("unified framework for forecasting non-stationary time series.")